In [11]:
import pandas as pd
import os
from pathlib import Path

# === Input configuration ===
table_path = Path("res_init2table.tsv")

# === Load the TSV file into a MultiIndexed DataFrame ===
df = pd.read_csv(table_path, sep='\t', index_col=[0, 1, 2, 3])
idx = df.index

# === Define filters for specific atoms by category ===
o3_filter = (idx.get_level_values("category") == "dna3term") & (idx.get_level_values("atom_name") == "O3'")
p_filter = (idx.get_level_values("category") == "dna5term") & (idx.get_level_values("atom_name") == "P")
o_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "O")
h1_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "H1")
h2_filter = (idx.get_level_values("category") == "nuc") & (idx.get_level_values("atom_name") == "H2")
nd1_filter = (idx.get_level_values("category") == "base") & (idx.get_level_values("atom_name") == "ND1")

# === Combine and extract relevant entries ===
final_mask = o3_filter | p_filter | o_filter | h1_filter | h2_filter | nd1_filter
atoms_df = df[final_mask]
atoms_idx = atoms_df.index

# === Organize atom lookups by atom name ===
def get_atoms(atom_name):
    return atoms_df.loc[atoms_idx.get_level_values("atom_name") == atom_name].index.tolist()

o3_atoms = get_atoms("O3'")
p_atoms = get_atoms("P")
o_atoms = get_atoms("O")
h1_atoms = get_atoms("H1")
h2_atoms = get_atoms("H2")
nd1_atoms = get_atoms("ND1")

# === Define desired interaction pairs ===
def pair_masks(atom_list1, atom_list2):
    pairs = []
    for _, _, resid1, atom1 in atom_list1:
        for _, _, resid2, atom2 in atom_list2:
            pairs.append(f":{resid1}@{atom1} :{resid2}@{atom2}")
    return pairs

# O3' to P
o3_p_pairs = pair_masks(o3_atoms, p_atoms)
# P to O
p_o_pairs = pair_masks(p_atoms, o_atoms)
# O - H1, O - H2
o_h1_pairs = pair_masks(o_atoms, h1_atoms)
o_h2_pairs = pair_masks(o_atoms, h2_atoms)
# H1 - ND1, H2 - ND1
h1_nd1_pairs = pair_masks(h1_atoms, nd1_atoms)
h2_nd1_pairs = pair_masks(h2_atoms, nd1_atoms)

# Combine all pairs
all_pairs = o3_p_pairs + p_o_pairs + o_h1_pairs + o_h2_pairs + h1_nd1_pairs + h2_nd1_pairs

# === Save to file ===
def save_rxn_mask(lines):
    try:
        script_name = os.path.splitext(os.path.basename(__file__))[0]
    except NameError:
        script_name = "get_rxn"
    output_filename = f"{script_name}.tsv"
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("Atom Pairs\n")
        f.write("\n".join(lines))

save_rxn_mask(all_pairs)
print(all_pairs)

[":13@O3' :14@P", ':14@P :729@O', ':729@O :729@H1', ':729@O :729@H2', ':729@H1 :303@ND1', ':729@H2 :303@ND1']
